# درس ۱۸ (ادامه): رسیدهایی که اثبات می‌کنند یک *انسان* مجوز انجام عمل را صادر کرده است  

این درس ثابت می‌کند که **عامل** چه کاری انجام داد و **دروازه** چه تصمیمی گرفت. این دفترچه نصف گمشده را اضافه می‌کند: اثبات اینکه یک **انسان نام‌دار** دقیقاً همان عمل را تأیید کرده است — یک امضای جداگانه که در اختیار انسان است بر روی کل عمل استاندارد، که به صورت آفلاین تأیید می‌شود.  

هر دو اثر در اینجا از **نقش پاکتی یکسان با رسیدهای درس** استفاده می‌کنند: یک بار مفید صاف با فیلد `type` که توسط Ed25519 مستقیماً بر بایت‌های استاندارد JCS امضا شده است، با شیء ساختاری `signature` متصل شده (و از بایت‌های امضا شده مستثنی شده). رسید تأیید یک `type` جدید (`human.approval.v1`) در کنار نوع عمل است، بنابراین یک تابع `verify_chain` هر دو نوع اثر را با همان مسیر کدی که در دفترچه اصلی ساختید پوشش می‌دهد. این رسید تأیید انسانی یک ترکیب آموزشی تعریف شده در اینجا است، نه نوع رسیدی که توسط [draft-farley-acta-signed-receipts] تعریف شده باشد.  

یک ارتقاء عمدی نسبت به تاییدکننده نمایشی در دفترچه اصلی: تاییدکننده اینجا به جای اینکه به کلید عمومی داخل رسید اعتماد کند، `signature.key_id` را در برابر یک **رجیستری کلید پین شده** حل می‌کند. این رویکرد تولیدی همان است که چک‌لیست خود درس توصیه می‌کند ("کلید عمومی تایید را منتشر کنید") و همین باعث می‌شود جعل تبدیل به رد کردن شود نه راه انداختن دور زدن با کلید خودتان.  

قاعده‌ای که این دفترچه آموزش می‌دهد: **یک تأیید امضا شده به تنهایی مجوز محسوب نمی‌شود.** مجوز فقط در صورتی وجود دارد که رسید تأیید و رسید عمل در زمان اجرا به همان عمل استاندارد متصل باشند، تحت نسخه سیاست، کلید، و انقضایی که هنوز معتبر است، و تأییدی که قبلاً مصرف نشده باشد. هر شکست با یک **دلیل متمایز** رد می‌شود، بنابراین شما می‌توانید بفهمید که *مجوز منقضی شده* از *عمل اجرا شده تغییر کرده* جدا است.  


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## عمل دقیق

واحد تأیید، **شیء عمل اصلی (canonical action object)** است — نه یک برچسب مبهم مانند «تأیید بازپرداخت»، بلکه عمل دقیق و کاملاً مشخص شده. امضای کل شیء (و استخراج چکیده از آن) چیزی است که به ما امکان می‌دهد بعداً ثابت کنیم که انسان *این* را تأیید کرده است و هیچ چیز دیگری را.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## یک پاکت، دو مرجع

هر رسید، پاکت درس است: یک داده ساده با فیلد `type`، به علاوه شیء `signature` (`alg`، `sig`، `key_id`) که بخشی از بایت‌های امضا شده نیست. `verify_envelope` بررسی ساختاری و امضا مشترک برای هر دو نوع رسید است؛ که کدام **ثبت کلید پین‌شده** برای حل مقدار `signature.key_id` استفاده می‌شود، مرجع‌ها را از هم جدا نگه می‌دارد:

- **رسید تاییدیه** (`human.approval.v1`) — تاییدکننده نام‌دار، عمل رسمی کامل **و خلاصه‌اش**، `policy_version`، زمان‌های صدور و انقضاء. مصرف یک‌باره در سطح زنجیره پیگیری می‌شود.
- **رسید عمل** (`agent.action.v1`) — هویت عامل، `run_id`، همان خلاصه عمل رسمی **، نتیجه اجرا و زمان، و `parent_approval_ref`: `receipt_hash` تاییدیه، همان قاعده مشابه `previous_receipt_hash` در زنجیره درس.

فیلد مشترک `action_digest` پیوندی است که اتصال بستگی به آن دارد. `key_id` فقط به عنوان راهنمای جستجو در شیء امضا قرار دارد: تغییر آن به کلید پین‌شده متفاوت باعث شکست بررسی امضا می‌شود، پس هیچ ویژگی‌ای منتقل نمی‌کند.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: جایی که پیوند در واقع تصمیم‌گیری می‌شود

`verify_chain` یک بسته‌بندی راحت بر روی دو بررسی امضا **نیست**. این تنها جایی است که `action_digest` مشترک و قانونی، تازه بودن سیاست/کلید/انقضای تاییدیه و مصرف یک‌باره تاییدیه به طور همزمان، در برابر عملی که *هم‌اکنون* اجرا می‌شود، بررسی می‌شوند.

هر شکست با یک **دلیل متمایز** رد می‌شود، بنابراین خواننده رد شدن می‌تواند تشخیص دهد که آیا صلاحیت منقضی شده است (سیاست تغییر کرده، کلید چرخانده شده، تاییدیه منقضی یا مصرف شده) یا عملی که اجرا شده است از زیر یک تاییدیه هنوز معتبر تغییر کرده است (تعویض digest).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## آنچه بایندینگ می‌گیرد

هر مورد زیر با **بستن** و با یک **دلیل متمایز** شکست می‌خورد. بلوک اول مجموعه کلاسیک است (دستکاری، نماینده گیج‌شده، پخش مجدد، جعل در هر دو اختیار، ورودی نادرست). بلوک دوم جفتی است که خاصیت را واقعی می‌کند نه صرفاً اثبات شده:

- **اختیار قدیمی** — امضاء هنوز معتبر است، اما نسخه سیاست تغییر کرده، کلید تاییدکننده از ثبت‌نام پین‌شده خارج شده، یا تایید قبل از اجرا منقضی شده است؛
- **جانشینی خلاصه** — یک رسید اقدام به‌درستی امضاء شده که `parent_approval_ref` آن به یک تایید واقعی اشاره می‌کند، اما خلاصه اقدام مرسوم آن تایید با عملی که واقعاً اجرا می‌شود مطابقت ندارد.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## این چه چیزی را اثبات می‌کند — و چه چیزی را اثبات نمی‌کند

**اثبات می‌کند:** یک انسان نام‌دار این *عمل دقیقاً کانونی‌شده* (عمل کامل + خلاصه، امضا شده با کلیدی که از یک رجیستری پین شده استخراج شده) را تأیید کرده است، و عامل دقیقاً همان عمل تأیید شده را اجرا کرده است (همان خلاصه، رسیدی که توسط `receipt_hash` به تأیید متصل شده است، طبق قرارداد زنجیره‌ای درس) — در حالی که نسخه سیاست، کلید و انقضای تأیید هنوز معتبر بودند، دقیقاً یک بار. اگر هر طرف تغییر کند، زنجیره بسته می‌شود و دلیل رد به شما می‌گوید **کدام** ویژگی شکسته شده است: مرجع منسوخ در مقابل عملی که تغییر کرده است.

**اثبات نمی‌کند:** اینکه رابط کاربری تأیید به انسان نشان داده است همان چیزی را که فکر می‌کرد امضا می‌کند (WYSIWYS مشکل خودش است)، اینکه کلید قبل از چرخش اجبار یا دزدیده نشده است، یا اینکه اثرات پایین‌دستی مطابق با عمل بوده‌اند. امضا شده ≠ مجاز شده: امضای معتبر بر یک سیاست منسوخ، کلید چرخشی، بازه زمانی منقضی یا خلاصه‌ای متفاوت در اینجا هیچ اعتبار نمی‌دهد.

دو نوع رسید به عمد پاکت درس و یک مسیر کد `verify_chain` مشترک دارند: پیوندی که برای رسیدهای عمل در دفتر اصلی ساختید همان کد است که تأیید انسان را بررسی می‌کند. یک قرارداد تاییدکننده، مراجع پین شده جداگانه، به هم پیوسته توسط خلاصه عمل کانونی و هیچ چیز دیگر.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
